# 운전 콘솔 앱 (OOP)

## 객체지향프로그래밍 OOP
- 현실세계의 모든 사건은 객체(object)와 객체사이의 상호작용인 것에 주목한다.
- 객체간의 상호작용을 구현하기 위해 의인화가 적용된다.
- 객체는 스스로 책임을 가지고 행동을 수행할 수 있다.
- 객체는 적절한 책임을 가질수 있도록 클래스 단위로 잘 분리해 작성해야 한다.
- 단일책임원칙 SRP Single Responsibility Principle 하나의 객체는 하나의 책임만 진다.
- (SOLID 객체지향 개발원칙을 참고)

## 객체간의 상호작용
- 객체는 아는 것(속성)과 할수 있는 것(메서드)으로 구성된다.
- 객체는 자기 할수 있는 것을 메서드로써 외부에 노출한다.
- 객체와 객체는 메세지를 주고받는다.
- 송신자객체는 수신자객체에 매개인자를 통해 메소드를 호출함으로써 메시지를 전송한다.
- 수신자객체는 일련의 책임을 수행후 리턴값으로 송신자객체에 응답한다.

## Driving Application

### 1. 요구사항 명세 (클라이언트)
- 운전프로그램을 만들어 주세요.
- 운전자는 시동걸기/끄기, 악셀 또는 브레이크를 밟을 수 있습니다.
- 자동차는 엔진시작/끝, 가속/감속을 할 수 있습니다.
- 자동차는 처음에 대기상태 있어야 합니다. (시동이 꺼진 상태)
- 자동차는 운전자에 의해 시동이 걸리고, 이미 시동이 걸려있다면, 또 시동을 걸수는 없습니다.
- 운전자가 악셀을 밟으면, 자동차는 10km/h씩 가속할 수 있습니다.
- 최대속도는 200km/h 입니다.
- 운전자가 브레이크를 밟으면, 자동차는 10km/h씩 감속할 수 있습니다.
- 운전자가 시동을 끄면, 자동차는 더이상 움직일 수 없습니다. (시동이 꺼진 상태)
- 자동차가 달리는 동안에는 시동을 끌수 없습니다.

### 2. 객체 도출
- 운전자
- 자동차
- 프로그램메뉴 (사용자 보게될 메뉴, 입력폼, 결과출력 담당할 UI객체)

### 3. 구현 포인트

요구사항은 단순히 속도만 올리고 내리는 것이 아니라, 현재 자동차의 상태를 확인하면서 동작을 제한해야 한다.

- 시동이 꺼져 있으면 가속/감속 불가
- 이미 시동이 걸려 있으면 다시 시동 걸기 불가
- 속도가 0보다 크면 시동 끄기 불가
- 최대 속도를 초과하지 않도록 제한

노트북에서는 `input()`으로 실행을 멈추지 않도록 샘플 메뉴 번호를 전달해 실행한다.


In [13]:
class Car:
    SPEED_STEP = 10
    MAX_SPEED = 200

    def __init__(self):
        self.__engine_started = False
        self.__speed = 0

    def start_engine(self):
        if self.__engine_started:
            return False, '이미 시동이 걸려 있습니다.'

        self.__engine_started = True
        return True, '부릉~ 시동이 정상적으로 걸렸습니다.'

    def stop_engine(self):
        if not self.__engine_started:
            return False, '이미 시동이 꺼져 있습니다.'

        if self.__speed > 0:
            return False, '주행 중에는 시동을 끌 수 없습니다.'

        self.__engine_started = False
        return True, '시동을 껐습니다.'

    def increase_speed(self):
        if not self.__engine_started:
            return self.__speed, '시동이 꺼져 있어 가속할 수 없습니다.'

        if self.__speed >= Car.MAX_SPEED:
            return self.__speed, '최대 속도입니다.'

        self.__speed += Car.SPEED_STEP
        return self.__speed, '가속했습니다.'

    def decrease_speed(self):
        if not self.__engine_started:
            return self.__speed, '시동이 꺼져 있어 감속할 수 없습니다.'

        if self.__speed <= 0:
            return self.__speed, '이미 정지 상태입니다.'

        self.__speed -= Car.SPEED_STEP
        return self.__speed, '감속했습니다.'


In [14]:
class Driver:

    def __init__(self):
        self.__car = Car()

    def start_car(self):
        return self.__car.start_engine()

    def stop_car(self):
        return self.__car.stop_engine()

    def accelerate(self):
        return self.__car.increase_speed()

    def brake(self):
        return self.__car.decrease_speed()


In [15]:
class Menu:

    def __init__(self):
        self.__driver = Driver()

    def main_menu(self, choices: list[str] | None = None):
        menu = """
----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------
"""
        choice_iter = iter(choices) if choices is not None else None

        while True:
            print(menu)

            if choice_iter is None:
                choice = input('메뉴 번호 입력: ')
            else:
                try:
                    choice = next(choice_iter)
                    print('샘플 메뉴 번호:', choice)
                except StopIteration:
                    choice = '0'
                    print('샘플 메뉴 번호:', choice)

            match choice:
                case '1':
                    result, message = self.__driver.start_car()
                    print('시동 걸기 결과:', message)
                case '2':
                    result, message = self.__driver.stop_car()
                    print('시동 끄기 결과:', message)
                case '3':
                    current_speed, message = self.__driver.accelerate()
                    print('가속 결과:', message)
                    print('현재 속도:', f'{current_speed}km/h')
                case '4':
                    current_speed, message = self.__driver.brake()
                    print('감속 결과:', message)
                    print('현재 속도:', f'{current_speed}km/h')
                case '0':
                    break
                case _:
                    print('잘못 누르셨습니다.')

        print('이용해 주셔서 감사합니다.')


In [16]:
# 실행
# 실제 수업에서는 menu.main_menu()로 실행하면 input()을 받을 수 있다.
# 노트북 자동 실행 검증을 위해 여기서는 샘플 메뉴 번호를 전달한다.
menu = Menu()
menu.main_menu(None)



----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------

시동 걸기 결과: 부릉~ 시동이 정상적으로 걸렸습니다.

----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------

시동 끄기 결과: 시동을 껐습니다.

----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------

시동 걸기 결과: 부릉~ 시동이 정상적으로 걸렸습니다.

----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------

가속 결과: 가속했습니다.
현재 속도: 10km/h

----------------------------------
Driving Application
----------------------------------
1. 시동 걸기
2. 시동 끄기
3. 악셀 밟기
4. 브레이크 밟기
0. 프로그램 종료
----------------------------------

가속 결과: 가속